# 04 — Dark Current Components  


---
## 1. Imports

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils.ingestion import DataIngestionConfig, DataIngestionService
from src.core.constants import q, kB, VT

print('Imports OK')

---
## 2. Build device and simulator

In [ ]:
config = DataIngestionConfig.from_defaults()
svc = DataIngestionService(config)
sim = svc.build_simulator()
device = sim.device

# Find breakdown
Vbr, _ = sim.find_breakdown(V_start=20, V_max=80, V_step=1.0)
print(f"Vbr = {Vbr:.1f} V")
print(f"Dark current components: {[type(c).__name__ for c in sim.current.components]}")

---
## 3. Dark current generation mechanisms

### 3.1 SRH generation

Shockley-Read-Hall generation via mid-gap traps:
$$
G_{\text{SRH}} = \frac{n_i}{2\tau}
$$

This is the dominant mechanism in the **low-field absorber** region. It depends on temperature through $n_i(T) \propto \exp(-E_g/2kT)$.

In [ ]:
# Compute SRH generation across device
phi, E, _ = sim.solve_poisson(Vbr - 5)  # 5 V below breakdown
x = device.grid.x
x_um = x * 1e4
E_abs = np.abs(E)

# SRH generation rate per component
srh_comp = sim.current.components[0]  # SRH is first
try:
    J_srh = srh_comp.compute(x, E_abs)
    print(f"SRH component: {type(srh_comp).__name__}")
except Exception as e:
    print(f"SRH compute failed: {e}")
    J_srh = np.ones_like(x) * 1e-15

plt.figure(figsize=(10, 3))
plt.semilogy(x_um, np.abs(J_srh), 'b-', lw=2)
plt.xlabel('Depth (µm)')
plt.ylabel('SRH current density (A/cm²)')
plt.grid(alpha=0.3)
plt.title('SRH generation across device')
plt.show()

### 3.2 Band-to-band tunneling (BTBT)

Kane model for direct tunneling in high-field regions:
$$
G_{\text{BTBT}} = A_{\text{BTBT}} \cdot E^2 \cdot \exp\!\left(-\frac{B_{\text{BTBT}}}{E}\right)
$$

BTBT is significant only where the field exceeds $\sim 10^6$ V/cm — inside the multiplication region near the charge sheet.

In [ ]:
# BTBT parameters (InP)
A_btbt = 2.84e15  # cm⁻¹·V⁻²·s⁻¹
B_btbt = 2.3e7    # V/cm

G_btbt = A_btbt * E_abs**2 * np.exp(-B_btbt / (E_abs + 1e4))
G_btbt[E_abs < 1e5] = 0  # negligible at low field

plt.figure(figsize=(10, 3))
plt.semilogy(x_um, G_btbt, 'r-', lw=2, label='BTBT gen. rate')
plt.semilogy(x_um, E_abs / E_abs.max(), 'k--', alpha=0.4, label='|E| (norm)')
plt.xlabel('Depth (µm)')
plt.ylabel('BTBT rate (cm⁻³·s⁻¹)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('BTBT generation — significant only in high-field region')
plt.show()

### 3.3 Trap-assisted tunneling (TAT)

Hurkx phonon-assisted tunneling model. Carriers tunnel through the bandgap via a trap state, enhanced by the electric field:

$$
G_{\text{TAT}} = \frac{n_i}{2\tau} \cdot (\Gamma_e + \Gamma_h)
$$

The Hurkx enhancement factor:
$$
\Gamma = \frac{E_B}{kT} \sqrt{\frac{\pi}{2}} \frac{F}{F_0} \exp\!\left(\frac{F^2}{2F_0^2}\right) \operatorname{erfc}\!\left(-\frac{F}{\sqrt{2}F_0}\right), \quad
F_0 = \frac{4\sqrt{2m^*}E_B^{3/2}}{3q\hbar}
$$

In [ ]:
# TAT enhancement factor vs field
from scipy import special
F_range = np.logspace(4, 6.5, 200)  # V/cm
T = 300.0
m_star = 0.2  # effective mass (relative)
hbar = 1.0545718e-34
eV_to_J = 1.602176634e-19

def gamma_hurkx(F, Eb_ev=0.75):
    Eb_J = Eb_ev * eV_to_J
    m0 = 9.10938356e-31
    m_eff = m_star * m0
    F0 = (4 * np.sqrt(2 * m_eff) * Eb_J**1.5) / (3 * float(q.magnitude) * hbar)
    return (Eb_J / (float(kB.magnitude) * T) * np.sqrt(np.pi/2) * F / F0
            * np.exp(F**2 / (2 * F0**2)) * special.erfc(-F / (np.sqrt(2) * F0)))

G_tat_03 = gamma_hurkx(F_range, 0.3)
G_tat_05 = gamma_hurkx(F_range, 0.5)
G_tat_075 = gamma_hurkx(F_range, 0.75)

plt.figure(figsize=(8, 4))
plt.loglog(F_range, G_tat_03, lw=2, label='$E_B$ = 0.3 eV')
plt.loglog(F_range, G_tat_05, lw=2, label='$E_B$ = 0.5 eV')
plt.loglog(F_range, G_tat_075, lw=2, label='$E_B$ = 0.75 eV')
plt.xlabel('Electric Field (V/cm)')
plt.ylabel('TAT enhancement $\\Gamma$')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('Hurkx TAT enhancement factor')
plt.show()

---
## 4. Total dark current vs bias

The simulator combines all three mechanisms and multiplies by $M(V)$ to get the total dark current:  
$I_{\text{dark}}(V) = \left[\int (J_{\text{SRH}} + J_{\text{BTBT}} + J_{\text{TAT}})\,dx\right] \times A \times M(V)$

In [ ]:
# Compute dark current at several biases
biases = np.arange(20.0, Vbr + 10, 2.0)
I_dark_list = []

for V in biases:
    try:
        dc = sim.compute_dark_current(float(V))
        I_dark_list.append(dc["I_dark"])
    except Exception as e:
        print(f"  V={V:.0f} failed: {e}")
        I_dark_list.append(np.nan)

plt.figure(figsize=(8, 4))
plt.semilogy(biases, np.abs(I_dark_list), 'o-', lw=2)
plt.axvline(Vbr, color='r', ls='--', alpha=0.5, label=f'Vbr = {Vbr} V')
plt.xlabel('Bias (V)')
plt.ylabel('Dark current (A)')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Total dark current vs bias')
plt.show()

---
## 5. Temperature dependence

DCR rises exponentially with temperature because:
1. $n_i(T) \propto \exp(-E_g/2kT)$ — SRH and TAT both scale with $n_i$
2. $V_{BR}(T)$ increases — at fixed excess bias, the effective field decreases slightly

In [ ]:
# ni vs temperature
temps = np.arange(250, 400, 5)
inp = device.materials['InP']
ni_vals = [inp.ni(T) for T in temps]

plt.figure(figsize=(8, 3))
plt.semilogy(temps, ni_vals, 'o-', lw=2)
plt.xlabel('Temperature (K)')
plt.ylabel('$n_i$ (cm⁻³)')
plt.grid(alpha=0.3)
plt.title('Intrinsic carrier density in InP vs temperature')
plt.show()

---
## 6. Summary

- **SRH**: Dominant in the low-field absorber; scales with $n_i(T)$.
- **BTBT**: Only matters in the high-field multiplication region; follows Kane's model.
- **TAT**: Trap-assisted tunneling with Hurkx field enhancement; bridges SRH and BTBT in moderate fields.
- The total dark current = $(J_{\text{SRH}} + J_{\text{BTBT}} + J_{\text{TAT}}) \times A \times M(V)$.
- DCR rises exponentially with temperature via $n_i(T)$.

**Next experiment (05):** Trigger probability and photon detection efficiency — the probability that a single photon triggers an avalanche.